# 19. Operator Fusion Introduction | 算子融合导论

**难度：** Medium | **环境：** CPU-first | **标签：** `编译系统`, `算子融合`, `性能优化` | **目标人群：** 编译与系统入门者

---

## 本节导读

算子融合通过减少中间张量落地、设备内存往返和计算内核启动，改变一段计算的数据路径。本节从未融合链路出发，判断哪些中间结果可以在片上继续消费，以及融合后必须保留哪些正确性条件。

完成后，你应能比较融合前后的内存流量和启动次数，并在形状、数值、后端支持与回退条件满足时，把融合作为候选而不是默认结论。

**关键词：** 算子融合（fusion）、内存流量、中间张量

![算子融合前后的数据路径与验证门槛](../docs/public/01_Hardware_Math_and_Systems/19_operator_fusion_data_path.svg)

---

## 前置阅读

**导语：** 先把执行模型和 block 级编程对齐，再看算子融合为什么能减少中间结果开销会更顺。

- [Group 1D: Execution Models, Asynchrony and Operator Mapping | 1D: 执行模型、异步调度与算子映射](./1D.md)
- [Group 1E: Compilers and Backends | 1E: 编译与后端](./1E.md)
- [18. Triton Block Model | Triton Block 模型](./18_Triton_Block_Model.ipynb)


## Q1：为什么算子之间的中间结果会这么贵？


中间结果贵，不只是因为它存在，而是因为它往往要被写回内存、再被下一步算子读回来。

如果一个计算图拆成很多小算子，每个算子都产生临时张量，那么这些张量就会频繁地在计算单元和内存之间往返。这样一来，真正耗时的可能不是计算，而是 memory traffic。

所以算子融合的第一层价值，是减少中间张量的落地次数。少一次写回、少一次读取，通常就意味着更少的带宽压力和更高的吞吐。


### Q1小验证：中间张量为什么拖慢系统

先记住“写回 + 再读回”这件事本身就很贵。

In [ ]:
def fusion_traffic(num_ops, tensor_bytes, fused_groups):
    """估算算子组边界造成的中间张量写回和重读字节数。"""
    if num_ops < 1 or not 1 <= fused_groups <= num_ops:
        raise ValueError('需要满足 1 <= fused_groups <= num_ops')
    if tensor_bytes <= 0:
        raise ValueError('tensor_bytes 必须大于 0')
    materialized_boundaries = fused_groups - 1
    intermediate_bytes = 2 * materialized_boundaries * tensor_bytes
    return {
        'launches': fused_groups,
        'materialized_boundaries': materialized_boundaries,
        'intermediate_bytes': intermediate_bytes,
    }

tensor_bytes = 64 * 1024**2
unfused = fusion_traffic(num_ops=4, tensor_bytes=tensor_bytes, fused_groups=4)
partially_fused = fusion_traffic(num_ops=4, tensor_bytes=tensor_bytes, fused_groups=2)
fully_fused = fusion_traffic(num_ops=4, tensor_bytes=tensor_bytes, fused_groups=1)
print({'unfused': unfused, 'partial': partially_fused, 'full': fully_fused})

assert unfused['intermediate_bytes'] == 6 * tensor_bytes
assert fully_fused['intermediate_bytes'] == 0


## Q2：算子融合为什么能改善数据局部性？


融合把原本分开的多个步骤合成一条更连续的执行链。

这样做的结果是：
- 中间结果更容易留在片上；
- 同一批数据更容易被连续复用；
- 数据搬运和计算之间的间隔更短。

数据局部性提升后，硬件就更容易把已有数据重复利用起来，而不是每一步都重新去 HBM 找一遍。这就是为什么融合不只是“少几个函数”，而是直接改变了数据流路径。


### Q2小验证：局部性为什么和融合有关

把“连续使用同一批数据”这件事记牢，就能理解融合的核心收益。

In [ ]:
def onchip_reuse(tile_bytes, reuse_count, onchip_budget_bytes):
    """判断 tile 能否驻留片上，并计算避免的设备内存读取量。"""
    if tile_bytes <= 0 or reuse_count < 1 or onchip_budget_bytes <= 0:
        raise ValueError('tile、复用次数和片上预算必须为正数')
    fits = tile_bytes <= onchip_budget_bytes
    avoided_read_bytes = tile_bytes * (reuse_count - 1) if fits else 0
    return {
        'fits_onchip': fits,
        'avoided_read_bytes': avoided_read_bytes,
        'budget_utilization': tile_bytes / onchip_budget_bytes,
    }

plans = {
    'small_tile': onchip_reuse(32 * 1024, 3, 96 * 1024),
    'large_tile': onchip_reuse(128 * 1024, 3, 96 * 1024),
}
for name, result in plans.items():
    print(name, '->', result)

assert plans['small_tile']['avoided_read_bytes'] == 64 * 1024
assert plans['large_tile']['avoided_read_bytes'] == 0


## Q3：为什么融合和编译器、kernel 实现会绑定在一起？


融合不是简单地把两个函数手动拼起来，而是需要在编译和实现层上重新安排计算顺序、临时变量和内存访问。

编译器能帮助决定哪些操作可以合并，kernel 实现则决定合并后如何在硬件上真正落地。比如：
- 哪些中间值可以直接在寄存器或 shared memory 中使用；
- 哪些步骤可以不必回到 HBM；
- 哪些访存模式可以一起优化。

所以融合是一个“编译决策 + kernel 实现”的组合问题。理解这一点后，后面看 Triton、CUDA 或 AI compiler 里的 fusion 逻辑就不会把它当成纯术语。


### Q3小验证：融合到底要改什么

先想内存路径，再想代码结构。

In [ ]:
def fusion_decision(semantics_match, shape_supported, onchip_fit, backend_lowered, reorder_bytes=0):
    """先检查正确性和执行硬门槛，再返回未通过原因。"""
    checks = {
        'semantics_match': semantics_match,
        'shape_supported': shape_supported,
        'onchip_fit': onchip_fit,
        'backend_lowered': backend_lowered,
        'no_extra_reorder': reorder_bytes == 0,
    }
    failures = [name for name, passed in checks.items() if not passed]
    return {'decision': 'benchmark' if not failures else 'defer', 'failures': failures}

cases = {
    'ready': (True, True, True, True, 0),
    'shape_miss': (True, False, True, True, 0),
    'reorder_cost': (True, True, True, True, 8 * 1024**2),
}
for name, config in cases.items():
    print(name, '->', fusion_decision(*config))

assert fusion_decision(*cases['ready'])['decision'] == 'benchmark'
assert 'no_extra_reorder' in fusion_decision(*cases['reorder_cost'])['failures']


## ⚠️ 常见误区

- 算子融合不是让代码变长，而是让中间结果少落地。
- 融合的目标不是“看上去更高级”，而是减少 memory traffic。
- 编译器和 kernel 都会参与融合，不能只看一侧。
- 先看数据流，再看代码名，通常更容易判断是否真的值得融合。

## 相关阅读

**导语：** 把算子融合继续接到性能调优、归约和 attention kernel，会更直观地看到中间张量是怎么被省掉的。

- [05. Triton 性能调优与基准测试 (Autotune & Profiling)](../03_Triton_Kernels/05_Triton_Autotune_and_Profiling.ipynb)
- [06. Triton 进阶：跨线程归约与数值稳定 (Safe Softmax)](../03_Triton_Kernels/06_Triton_Fused_Softmax.ipynb)
- [08. Triton Flash Attention | 真正的 Flash Attention 前向算子](../03_Triton_Kernels/08_Triton_Flash_Attention.ipynb)
---